Building an AI-powered document querying system that retrieves relevant sections from PDFs, generates accurate answers, and self-verifies them to prevent hallucinations

Frameworks & Libraries:

LangGraph — orchestrating the multi-agent workflow
Docling — extracting structured text from complex PDFs
ChromaDB — vector store for semantic search
BM25 — keyword-based retrieval (hybrid search)
Gradio — interactive web UI

Environment Setup

In [1]:
!pip install langgraph langchain langchain-community langchain-openai chromadb rank_bm25 docling sentence-transformers gradio

  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached python_pptx-1.0.2-py3-none-any.whl.metadata (2.5 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.me

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
logfire 4.33.0 requires opentelemetry-sdk<1.42.0,>=1.39.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
mistralai 2.4.5 requires opentelemetry-semantic-conventions<0.61,>=0.60b1, but you have opentelemetry-semantic-conventions 0.63b1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.39.1 requires opentelemetry-exporter-otlp-proto-common==1.39.1, but you have opentelemetry-exporter-otlp-proto-common 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.39.1 requires opentelemetry-proto==1.39.1, but you have opentelemetry-proto 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.39.1 requires opentelemetry-sdk~=1.39.1, but you have opentelemetry-sdk 1.42.1 which is incompatible.
opentelemetry-instrumentation 0.60b1 requires opentelemetry-semantic-conv

In [2]:
import langgraph
import chromadb
import gradio
import rank_bm25
print("All packages imported successfully!")

C:\Users\SANDEEP S\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All packages imported successfully!


In [3]:
import ollama


ollama.pull('llama3.1:8b')
print("Model download complete!")


Model download complete!


In [4]:
ollama.pull('nomic-embed-text')


ProgressResponse(status='success', completed=None, total=None, digest=None)

In [5]:
import requests

# Test Ollama is running
response = requests.get("http://localhost:11434/api/tags")
models = [m["name"] for m in response.json()["models"]]
print("Available models:", models)

Available models: ['nomic-embed-text:latest', 'llama3.1:8b', 'llava:latest', 'llama3.2:latest', 'qwen2.5:3b', 'qwen3.5:2b', 'gemma3:270m']


In [6]:
# Test LLM
import ollama

response = ollama.chat(
    model="llama3.1:8b",
    messages=[{"role": "user", "content": "Reply with just: LLM is working!"}]
)
print(response["message"]["content"])

LLM is working!


In [7]:
# Test Embedding model
response = ollama.embeddings(
    model="nomic-embed-text",
    prompt="test embedding"
)
print(f"✅ Embedding model working! Vector size: {len(response['embedding'])}")

✅ Embedding model working! Vector size: 768


Configuration Setup

In [8]:
import os

# ── LLM Configuration (Ollama) ─────────────────────────────────────
OLLAMA_BASE_URL = "http://localhost:11434"
LLM_MODEL = "llama3.1:8b"
EMBEDDING_MODEL = "nomic-embed-text"

# ── ChromaDB Configuration ─────────────────────────────────────────
CHROMA_COLLECTION_NAME = "docchat_collection"
CHROMA_PERSIST_DIR = "./chroma_db"

# ── Retriever Configuration ────────────────────────────────────────
TOP_K_RESULTS = 5
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# ── Agent Configuration ────────────────────────────────────────────
MAX_RETRIES = 3

# ── Data Directory ─────────────────────────────────────────────────
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)

print("✅ Config loaded!")

✅ Config loaded!


PDF Parsing with Docling (Single PDF)

In [9]:
from docling.document_converter import DocumentConverter

def load_document(pdf_path: str) -> list[dict]:
    """
    Converts a single PDF to clean text chunks using Docling.
    Returns a list of dicts with keys: text, source, chunk_id
    """
    print(f"📄 Processing: {pdf_path}")
    
    converter = DocumentConverter()
    result = converter.convert(pdf_path)
    markdown_text = result.document.export_to_markdown()
    
    chunks = chunk_text(markdown_text, source=pdf_path)
    print(f"✅ Extracted {len(chunks)} chunks")
    return chunks


def chunk_text(text: str, source: str, chunk_size=500, overlap=50) -> list[dict]:
    """Splits text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append({
                "text": chunk.strip(),
                "source": source,
                "chunk_id": len(chunks)
            })
        start = end - overlap
    return chunks

In [12]:
# Just point directly to your PDF
PDF_PATH = r"C:\Users\SANDEEP S\Desktop\casestudy.pdf"  # ← add the actual filename.pdf
chunks = load_document(PDF_PATH)

# Preview
print("\n--- Sample Chunk ---")
print(f"Source   : {chunks[0]['source']}")
print(f"Chunk ID : {chunks[0]['chunk_id']}")
print(f"Text     : {chunks[0]['text'][:300]}")

📄 Processing: C:\Users\SANDEEP S\Desktop\casestudy.pdf


[INFO] 2026-05-27 13:23:42,438 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 13:23:42,452 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-27 13:23:43,802 [RapidOCR] download_file.py:82: Download size: 4.53MB
[INFO] 2026-05-27 13:23:44,815 [RapidOCR] download_file.py:95: Successfully saved to: C:\Users\SANDEEP S\AppData\Local\Programs\Python\Python310\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-27 13:23:44,820 [RapidOCR] main.py:57: Using C:\Users\SANDEEP S\AppData\Local\Programs\Python\Python310\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-27 13:23:44,998 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 13:23:44,998 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4

✅ Extracted 5 chunks

--- Sample Chunk ---
Source   : C:\Users\SANDEEP S\Desktop\casestudy.pdf
Chunk ID : 0
Text     : CASE STUDY

## UNIFYING IT OPERATIONS: A SMARTER WAY TO TRACK NETWORK HEALTH

2x

PLATFORM PROCESSING SPEED

## CLIENT

A leading international branch campus enterprise managing a complex, multi-vendor network estate spanning wireless infrastructure, campus switching, perimeter firewalls, and power 


Step 5 — Hybrid Retriever (BM25 + ChromaDB)

Using TWO search methods and combine them:
1. BM25 (Keyword Search)

Works like a traditional search engine (like Ctrl+F but smarter)
Looks for exact or similar words from your question inside the chunks
Good at finding specific terms, names, numbers

2. ChromaDB (Semantic/Vector Search)

Understands the meaning of your question
Even if the exact words don't match, it finds chunks that are about the same topic
For example: you ask "cost savings" and it finds chunks talking about "budget reduction"

we will be combining both method cause:
BM25 misses chunks where words don't match exactly
1.Vector search sometimes misses very specific keyword matches
2.Together they cover each other's blind spots — this is called Hybrid Retrieval.

In [13]:
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
import ollama

# ── Embedding function using Ollama ────────────────────────────────
class OllamaEmbeddingFunction(embedding_functions.EmbeddingFunction):
    def __call__(self, input: list[str]) -> list[list[float]]:
        embeddings = []
        for text in input:
            response = ollama.embeddings(model="nomic-embed-text", prompt=text)
            embeddings.append(response["embedding"])
        return embeddings

# ── Setup ChromaDB ─────────────────────────────────────────────────
def setup_vectorstore(chunks: list[dict]):
    """Stores chunks in ChromaDB with Ollama embeddings."""
    
    embed_fn = OllamaEmbeddingFunction()
    
    client = chromadb.PersistentClient(path="./chroma_db")
    
    # Fresh collection each run for prototype
    try:
        client.delete_collection("docchat_collection")
    except:
        pass
    
    collection = client.create_collection(
        name="docchat_collection",
        embedding_function=embed_fn
    )
    
    collection.add(
        documents=[c["text"] for c in chunks],
        ids=[f"chunk_{c['chunk_id']}" for c in chunks],
        metadatas=[{"source": c["source"], "chunk_id": c["chunk_id"]} for c in chunks]
    )
    
    print(f"✅ Stored {len(chunks)} chunks in ChromaDB")
    return collection

# ── Setup BM25 ─────────────────────────────────────────────────────
def setup_bm25(chunks: list[dict]):
    """Builds a BM25 index from chunks."""
    tokenized = [c["text"].lower().split() for c in chunks]
    bm25 = BM25Okapi(tokenized)
    print("✅ BM25 index built")
    return bm25

# ── Hybrid Retriever ───────────────────────────────────────────────
def hybrid_retrieve(query: str, chunks: list[dict], collection, bm25, top_k=5) -> list[str]:
    """
    Combines BM25 (keyword) + ChromaDB (semantic) results.
    Returns deduplicated top_k chunks.
    """
    # BM25 retrieval
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:top_k]
    bm25_results = [chunks[i]["text"] for i in bm25_top_idx]

    # Vector retrieval
    vector_results = collection.query(query_texts=[query], n_results=top_k)
    vector_texts = vector_results["documents"][0]

    # Merge + deduplicate (preserve order)
    seen = set()
    combined = []
    for text in bm25_results + vector_texts:
        if text not in seen:
            seen.add(text)
            combined.append(text)

    return combined[:top_k]

In [14]:
# Build both indexes
collection = setup_vectorstore(chunks)
bm25 = setup_bm25(chunks)

# Test retrieval
query = "What is the main problem this case study solves?"
results = hybrid_retrieve(query, chunks, collection, bm25, top_k=3)

print(f"\n--- Top {len(results)} Retrieved Chunks ---")
for i, r in enumerate(results):
    print(f"\n[Chunk {i+1}]\n{r[:300]}")

C:\Users\SANDEEP S\AppData\Local\Temp\ipykernel_27764\3927732669.py:19: DeprecationWarning: The class OllamaEmbeddingFunction does not implement __init__. This will be required in a future version.
  embed_fn = OllamaEmbeddingFunction()


✅ Stored 5 chunks in ChromaDB
✅ BM25 index built

--- Top 3 Retrieved Chunks ---

[Chunk 1]
CASE STUDY

## UNIFYING IT OPERATIONS: A SMARTER WAY TO TRACK NETWORK HEALTH

2x

PLATFORM PROCESSING SPEED

## CLIENT

A leading international branch campus enterprise managing a complex, multi-vendor network estate spanning wireless infrastructure, campus switching, perimeter firewalls, and power 

[Chunk 2]
major international higher education institution operated a complex campus network spanning hardware from multiple distinct vendors.
- Each hardware family generated activity logs in its own format, scattering critical information across isolated tools.
- The infrastructure team lacked a unified env

[Chunk 3]
data bottlenecks as the campus network expanded.

## PROJECT OBJECTIVES

- Establish a centralized operational data platform to consolidate activity streams from all disparate hardware estates.
- Automate the identification and categorization of incoming data without requiring manual

Step 6 — Research Agent + Verification Agent

Agent 1 — Research Agent

Takes your question + the retrieved chunks as context
Sends both to llama3.1:8b with a strict instruction: "Answer ONLY from the context, don't guess"
Returns a grounded answer


Agent 2 — Verification Agent

Takes the question, the answer, and the original chunks
Sends all three to llama3.1:8b and asks: "Is this answer actually supported by the context?"
Returns one of two verdicts:

PASS — answer is grounded in the document
FAIL — answer contains something not in the document (hallucination detected)

In [15]:
import ollama

# ── Research Agent ─────────────────────────────────────────────────
def research_agent(query: str, context_chunks: list[str]) -> str:
    """
    Takes the query + retrieved chunks and generates an answer.
    """
    context = "\n\n".join(context_chunks)
    
    prompt = f"""You are a document research assistant. 
Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say "I could not find this information in the document."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:"""

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"]


# ── Verification Agent ─────────────────────────────────────────────
def verification_agent(query: str, answer: str, context_chunks: list[str]) -> dict:
    """
    Cross-checks the answer against the original retrieved chunks.
    Returns a dict with: verdict (PASS/FAIL) and reason.
    """
    context = "\n\n".join(context_chunks)
    
    prompt = f"""You are a fact-checking assistant.
You will be given a QUESTION, an ANSWER, and the SOURCE CONTEXT the answer was based on.

Your job is to verify whether the ANSWER is fully supported by the SOURCE CONTEXT.

Respond in this exact format:
VERDICT: PASS or FAIL
REASON: one sentence explaining your verdict

QUESTION: {query}

ANSWER: {answer}

SOURCE CONTEXT:
{context}"""

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response["message"]["content"]
    
    # Parse verdict and reason
    verdict = "PASS" if "VERDICT: PASS" in raw.upper() else "FAIL"
    reason = ""
    for line in raw.splitlines():
        if line.upper().startswith("REASON:"):
            reason = line.split(":", 1)[1].strip()
            break
    
    return {"verdict": verdict, "reason": reason, "raw": raw}

In [16]:
query = "What was the main problem this case study solves?"

# Step 1 — Research Agent generates answer
print(" Research Agent thinking...\n")
answer = research_agent(query, results)
print(f"ANSWER:\n{answer}")

# Step 2 — Verification Agent checks it
print("\n Verification Agent checking...\n")
verdict = verification_agent(query, answer, results)
print(f"VERDICT : {verdict['verdict']}")
print(f"REASON  : {verdict['reason']}")

 Research Agent thinking...

ANSWER:
The main problem this case study solves is that the infrastructure team lacked a unified environment to search, correlate, or trigger alerts across the disconnected systems of the complex campus network.

 Verification Agent checking...

VERDICT : PASS
REASON  : The SOURCE CONTEXT directly states that "The infrastructure team lacked a unified environment to search, correlate, or trigger alerts across these disconnected systems" as one of the problems faced by the client.


Step 7 — LangGraph Workflow (Self-Correction Loop)

Connect all the previous steps into one automated pipeline that runs start to finish on its own — and can retry automatically if something goes wrong.

In [17]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# Define the State
class DocChatState(TypedDict):
    query: str
    context_chunks: list[str]
    answer: str
    verdict: str
    reason: str
    retries: int

# Global references for nodes to access
chunks_global = chunks
collection_global = collection
bm25_global = bm25
MAX_RETRIES = 3

# Node 1 - Retrieve
def retrieve_node(state: DocChatState) -> DocChatState:
    print("[Node 1] Retrieving context...")
    retrieved = hybrid_retrieve(state["query"], chunks_global, collection_global, bm25_global, top_k=5)
    return {**state, "context_chunks": retrieved}

# Node 2 - Research
def research_node(state: DocChatState) -> DocChatState:
    print("[Node 2] Research Agent generating answer...")
    answer = research_agent(state["query"], state["context_chunks"])
    return {**state, "answer": answer}

# Node 3 - Verify
def verify_node(state: DocChatState) -> DocChatState:
    print("[Node 3] Verification Agent checking answer...")
    result = verification_agent(state["query"], state["answer"], state["context_chunks"])
    return {**state, "verdict": result["verdict"], "reason": result["reason"]}

# Node 4 - Increment Retry Counter
def increment_retry(state: DocChatState) -> DocChatState:
    return {**state, "retries": state["retries"] + 1}

# Decision Function - called after verify node
def should_retry(state: DocChatState) -> str:
    if state["verdict"] == "PASS":
        print("Verification passed!")
        return "end"
    elif state["retries"] < MAX_RETRIES:
        print(f"Verification failed. Retrying... (attempt {state['retries'] + 1}/{MAX_RETRIES})")
        return "retry"
    else:
        print("Max retries reached. Returning best available answer.")
        return "end"

# Build the Graph
def build_graph():
    graph = StateGraph(DocChatState)

    graph.add_node("retrieve", retrieve_node)
    graph.add_node("research", research_node)
    graph.add_node("verify", verify_node)
    graph.add_node("increment_retry", increment_retry)

    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve", "research")
    graph.add_edge("research", "verify")

    graph.add_conditional_edges(
        "verify",
        should_retry,
        {
            "end": END,
            "retry": "increment_retry"
        }
    )

    graph.add_edge("increment_retry", "research")

    return graph.compile()

In [18]:
app = build_graph()

query = "What was the main problem this case study solves?"

final_state = app.invoke({
    "query": query,
    "context_chunks": [],
    "answer": "",
    "verdict": "",
    "reason": "",
    "retries": 0
})

print("\n========== FINAL RESULT ==========")
print(f"QUESTION : {final_state['query']}")
print(f"ANSWER   : {final_state['answer']}")
print(f"VERDICT  : {final_state['verdict']}")
print(f"REASON   : {final_state['reason']}")

[Node 1] Retrieving context...
[Node 2] Research Agent generating answer...
[Node 3] Verification Agent checking answer...
Verification passed!

========== FINAL RESULT ==========
QUESTION : What was the main problem this case study solves?
ANSWER   : The main problem this case study solves is the lack of a unified environment for searching, correlating, and triggering alerts across multiple disconnected systems, which created visibility blind spots and risks of data bottlenecks as the campus network expanded.
VERDICT  : PASS
REASON   : The ANSWER is fully supported by the SOURCE CONTEXT, which describes the main problem as "the lack of a unified environment for searching, correlating, and triggering alerts across multiple disconnected systems", creating visibility blind spots and data bottlenecks.


Step 8 — Gradio UI

In [21]:
import gradio as gr

def run_docchat(pdf_path: str, query: str) -> tuple[str, str, str]:
    # Clean the path - remove any r" prefix and surrounding quotes if user typed them
    pdf_path = pdf_path.strip().strip('"').strip("'")
    if pdf_path.startswith('r'):
        pdf_path = pdf_path[1:].strip('"').strip("'")
    
    # Rest of the function stays exactly the same
    if not pdf_path.strip():
        return "Please provide a PDF path.", "", ""
    if not query.strip():
        return "Please enter a question.", "", ""

    chunks = load_document(pdf_path)
    if not chunks:
        return "Could not extract text from the PDF.", "", ""

    collection = setup_vectorstore(chunks)
    bm25 = setup_bm25(chunks)

    global chunks_global, collection_global, bm25_global
    chunks_global = chunks
    collection_global = collection
    bm25_global = bm25

    final_state = app.invoke({
        "query": query,
        "context_chunks": [],
        "answer": "",
        "verdict": "",
        "reason": "",
        "retries": 0
    })

    return (
        final_state["answer"],
        final_state["verdict"],
        final_state["reason"]
    )
# Build the UI
with gr.Blocks(title="DocChat") as demo:
    gr.Markdown("# DocChat - Multi-Agent RAG System")
    gr.Markdown("Upload a PDF path and ask questions. The system retrieves, answers, and verifies responses automatically.")

    with gr.Row():
        with gr.Column():
            pdf_input = gr.Textbox(
                label="PDF File Path",
                placeholder=r"C:\Users\yourname\Desktop\file.pdf"
            )
            query_input = gr.Textbox(
                label="Your Question",
                placeholder="What is this document about?",
                lines=2
            )
            submit_btn = gr.Button("Ask", variant="primary")

        with gr.Column():
            answer_output = gr.Textbox(label="Answer", lines=6)
            verdict_output = gr.Textbox(label="Verdict (PASS / FAIL)")
            reason_output = gr.Textbox(label="Verification Reason", lines=3)

    submit_btn.click(
        fn=run_docchat,
        inputs=[pdf_input, query_input],
        outputs=[answer_output, verdict_output, reason_output]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


📄 Processing: C:\Users\SANDEEP S\Desktop\casestudy.pdf


[INFO] 2026-05-27 13:46:04,369 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 13:46:04,393 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\SANDEEP S\AppData\Local\Programs\Python\Python310\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-27 13:46:04,398 [RapidOCR] main.py:57: Using C:\Users\SANDEEP S\AppData\Local\Programs\Python\Python310\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-27 13:46:04,578 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 13:46:04,589 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\SANDEEP S\AppData\Local\Programs\Python\Python310\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-27 13:46:04,592 [RapidOCR] main.py:57: Using C:\Users\SANDEEP S\AppData\Local\Programs\Python\Python310\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-27 13:46:04,692

✅ Extracted 5 chunks


C:\Users\SANDEEP S\AppData\Local\Temp\ipykernel_27764\3927732669.py:19: DeprecationWarning: The class OllamaEmbeddingFunction does not implement __init__. This will be required in a future version.
  embed_fn = OllamaEmbeddingFunction()


✅ Stored 5 chunks in ChromaDB
✅ BM25 index built
[Node 1] Retrieving context...
[Node 2] Research Agent generating answer...
[Node 3] Verification Agent checking answer...
Verification failed. Retrying... (attempt 1/3)
[Node 2] Research Agent generating answer...
[Node 3] Verification Agent checking answer...
Verification passed!
